In [12]:
import pandas as pd
import re
from io import StringIO

# 1) načtení raw textu
with open("dataset", "r", encoding="utf-8") as f:
    lines = f.readlines()

# 2) výběr názvů sloupců z @ATTRIBUTE
attribute_lines = [line.strip() for line in lines if line.strip().lower().startswith("@attribute")]

columns = []
for line in attribute_lines:
    match = re.match(r'@attribute\s+(".*?"|\S+)\s+(.+)$', line, flags=re.IGNORECASE)
    if match:
        col_name = match.group(1).strip('"')
        columns.append(col_name)

print("Sloupce:", columns)

# 3) nalezení začátku dat
data_index = None
for i, line in enumerate(lines):
    if line.strip().lower() == "@data":
        data_index = i
        break

# 4) vytažení pouze datových řádků
data_lines = []
for line in lines[data_index + 1:]:
    stripped = line.strip()
    if stripped and not stripped.startswith("%"):
        data_lines.append(stripped)

# 5) vytvoření DataFrame z dat
csv_text = "\n".join(data_lines)
df = pd.read_csv(StringIO(csv_text), header=None, names=columns)

# 6) kontrola dat
print(df.head())
print(df.info())
print(df["class"].value_counts())

# 7) volitelně uložit do CSV
df.to_csv("diabetes_dataset.csv", index=False)

Sloupce: ['Age', 'Gender', 'Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness', 'Polyphagia', 'Genital thrush', 'visual blurring', 'Itching', 'Irritability', 'delayed healing', 'partial paresis', 'muscle stiffness', 'Alopecia', 'Obesity', 'class']
   Age Gender Polyuria Polydipsia sudden weight loss weakness Polyphagia  \
0   40   Male       No        Yes                 No      Yes         No   
1   58   Male       No         No                 No      Yes         No   
2   41   Male      Yes         No                 No      Yes        Yes   
3   45   Male       No         No                Yes      Yes        Yes   
4   60   Male      Yes        Yes                Yes      Yes        Yes   

  Genital thrush visual blurring Itching Irritability delayed healing  \
0             No              No     Yes           No             Yes   
1             No             Yes      No           No              No   
2             No              No     Yes           No             Yes

In [13]:
print(df.head())
print(df.columns.tolist())
print(df["class"].value_counts())

   Age Gender Polyuria Polydipsia sudden weight loss weakness Polyphagia  \
0   40   Male       No        Yes                 No      Yes         No   
1   58   Male       No         No                 No      Yes         No   
2   41   Male      Yes         No                 No      Yes        Yes   
3   45   Male       No         No                Yes      Yes        Yes   
4   60   Male      Yes        Yes                Yes      Yes        Yes   

  Genital thrush visual blurring Itching Irritability delayed healing  \
0             No              No     Yes           No             Yes   
1             No             Yes      No           No              No   
2             No              No     Yes           No             Yes   
3            Yes              No     Yes           No             Yes   
4             No             Yes     Yes          Yes             Yes   

  partial paresis muscle stiffness Alopecia Obesity     class  
0              No              Yes      

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score

# 1) target na 0/1
df["class"] = df["class"].map({"Positive": 1, "Negative": 0})

# 2) X a y
X = df.drop(columns=["class"])
y = df["class"]

# 3) rozdělení sloupců
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Kategorické sloupce:", categorical_cols)
print("Numerické sloupce:", numeric_cols)

# 4) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 5) preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# 6) model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=4, random_state=42))
])

# 7) tréning
model.fit(X_train, y_train)

# 8) predikce
y_pred = model.predict(X_test)

# 9) evaluace
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Kategorické sloupce: ['Gender', 'Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness', 'Polyphagia', 'Genital thrush', 'visual blurring', 'Itching', 'Irritability', 'delayed healing', 'partial paresis', 'muscle stiffness', 'Alopecia', 'Obesity']
Numerické sloupce: ['Age']
Accuracy: 0.9519230769230769
F1: 0.959349593495935
Recall: 0.921875
Confusion matrix:
 [[40  0]
 [ 5 59]]
              precision    recall  f1-score   support

           0       0.89      1.00      0.94        40
           1       1.00      0.92      0.96        64

    accuracy                           0.95       104
   macro avg       0.94      0.96      0.95       104
weighted avg       0.96      0.95      0.95       104

